# Notebook 01 — PubMed Data Pipeline

This notebook explores the PubMed E-utilities API, prototypes the fetching logic,
and runs the full pipeline to build our article corpus.

**Parts covered:**
- API exploration: esearch + efetch
- XML parsing walkthrough
- Deduplication analysis
- Final pipeline run & corpus statistics

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
import requests
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

load_dotenv('../.env')
print('Setup complete')

## 1. API Exploration — esearch

The PubMed E-utilities API has two key endpoints:
- **esearch**: Find PMIDs matching a query
- **efetch**: Fetch full records for a list of PMIDs

We use `sort=pub_date` to get the most recent articles.

In [ ]:
ESEARCH_URL = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
EFETCH_URL = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'

# Test esearch for one term
term = 'type 2 diabetes mellitus'
params = {
    'db': 'pubmed',
    'term': term,
    'retmax': 5,
    'sort': 'pub_date',
    'retmode': 'json',
}
resp = requests.get(ESEARCH_URL, params=params)
data = resp.json()

print(f'Search: {term!r}')
print(f'Total results in PubMed: {data["esearchresult"]["count"]}')
print(f'PMIDs returned: {data["esearchresult"]["idlist"]}')

## 2. efetch — XML Structure Walkthrough

In [ ]:
import time

pmids = data['esearchresult']['idlist']
time.sleep(0.34)  # Rate limiting: 3 req/sec

fetch_params = {
    'db': 'pubmed',
    'id': ','.join(pmids[:2]),  # Just 2 for exploration
    'rettype': 'abstract',
    'retmode': 'xml',
}
fetch_resp = requests.get(EFETCH_URL, params=fetch_params)
root = ET.fromstring(fetch_resp.content)

# Show the XML structure of the first article
first = root.find('PubmedArticle')
medline = first.find('MedlineCitation')
article = medline.find('Article')

print('=== XML Structure Walkthrough ===')
print(f'PMID: {medline.find("PMID").text}')
print(f'Title: {article.find("ArticleTitle").text[:100]}...')

# Check for structured abstract (multiple AbstractText nodes)
abstract_el = article.find('Abstract')
if abstract_el is not None:
    ab_texts = abstract_el.findall('AbstractText')
    print(f'\nAbstract sections: {len(ab_texts)}')
    for ab in ab_texts:
        label = ab.get('Label', 'UNLABELED')
        text_preview = (ab.text or '')[:60]
        print(f'  [{label}]: {text_preview}...')

# DOI location
pubmed_data = first.find('PubmedData')
for article_id in pubmed_data.findall('ArticleIdList/ArticleId'):
    print(f'  ArticleId type={article_id.get("IdType")}: {article_id.text}')

## 3. Run the Full Pipeline

In [ ]:
from src.pipeline import run_pipeline
from pathlib import Path

articles = run_pipeline(
    csv_path=Path('../medical_terms.csv'),
    output_path=Path('../data/pubmed_articles.json'),
)
print(f'\nTotal articles in corpus: {len(articles)}')

## 4. Corpus Statistics

In [ ]:
import pandas as pd

# Convert to DataFrame for easy analysis
df = pd.DataFrame(articles)

print(f'Total articles: {len(df)}')
print(f'Articles with abstracts: {df["abstract"].apply(lambda x: len(x) > 10).sum()}')
print(f'Articles with DOI: {df["doi"].notna().sum()}')
print(f'Year range: {df["year"].min()} - {df["year"].max()}')
print(f'\nTop journals:')
print(df['journal'].value_counts().head(10))

print(f'\nMatched terms distribution:')
from collections import Counter
term_counts = Counter()
for art in articles:
    for term in art.get('matched_terms', []):
        term_counts[term] += 1
for term, count in sorted(term_counts.items(), key=lambda x: -x[1]):
    print(f'  {term}: {count} articles')

## 5. Deduplication Analysis

In [ ]:
# Articles that matched more than one term
multi_term = [a for a in articles if len(a.get('matched_terms', [])) > 1]
print(f'Articles matching multiple terms: {len(multi_term)}')
print()
for art in multi_term[:5]:
    print(f'  PMID {art["pmid"]}: {art["title"][:60]}...')
    print(f'    Terms: {art["matched_terms"]}')

## 6. Sample Article Display

In [ ]:
# Show one full article
sample = articles[0]
print(f'PMID       : {sample["pmid"]}')
print(f'Title      : {sample["title"]}')
print(f'Authors    : {sample["authors"]}')
print(f'Journal    : {sample["journal"]}')
print(f'Year       : {sample["year"]}')
print(f'DOI        : {sample["doi"]}')
print(f'Terms      : {sample["matched_terms"]}')
print(f'\nAbstract ({len(sample["abstract"])} chars):')
print(sample['abstract'][:500] + '...' if len(sample['abstract']) > 500 else sample['abstract'])